In [0]:
df = spark.table("bronze.bronze_raw")
df.display()

In [0]:
from pyspark.sql.functions import unix_timestamp


In [0]:
df_silver = (
    df
    .withColumn("pickup_datetime", df.tpep_pickup_datetime.cast("timestamp"))
    .withColumn("dropoff_datetime", df.tpep_dropoff_datetime.cast("timestamp"))
    .withColumn("trip_distance", df.trip_distance.cast("double"))
    .withColumn("trip_duration_minutes",(unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60)
    .withColumn("fare_amount", df.fare_amount.cast("double"))
    .withColumn("pickup_zip", df.pickup_zip.cast("string"))
    .withColumn("dropoff_zip", df.dropoff_zip.cast("string"))
    .filter("fare_amount > 0 AND trip_distance > 0")
    .filter("trip_duration_minutes IS NOT NULL AND trip_duration_minutes > 0")  
)

display(df_silver)


In [0]:
%sql
create schema if not exists silver

In [0]:
df_silver.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("silver.transformed_data")